## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [1]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT



### Load Translator resources


In [2]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources(use_new_metakg_url=True)

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


In [7]:
APInames

{'Clinical Trials KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/ctkp/query',
 'ARAX Translator Reasoner - TRAPI 1.6.0': 'https://arax.transltr.io/api/arax/v1.4/query/',
 'MolePro': 'https://molepro-trapi.transltr.io/molepro/trapi/v1.5/query/',
 'RTX KG2 - TRAPI 1.5.0': 'https://kg2cploverdb.ci.transltr.io/kg2c/query',
 'BioThings Explorer (BTE) TRAPI': 'https://bte.transltr.io/v1/query/',
 'Gene-List Network Enrichment Analysis': 'https://translator.broadinstitute.org/gelinea-trapi/v1.5/query/',
 'Autonomous Relay System (ARS) TRAPI': 'https://ars-prod.transltr.io/ars/api/submit/',
 'Automat-reactome(Trapi v1.5.0)': 'https://automat.renci.org/reactome/query/',
 'Multiomics KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/multiomics/query',
 'Drug Approvals KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/dakp/query',
 'Microbiome KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/mbkp/query',
 'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0': 'https://multiomics

### Select endpoints for query


In [24]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = ['Retriever',
                    #'Clinical Trials KP - TRAPI 1.5.0',
                    #'Drug Approvals KP - TRAPI 1.5.0',
                    #'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    #'Microbiome KP - TRAPI 1.5.0',
                    #'MolePro',
                    #'COHD TRAPI',
                    #'RTX KG2 - TRAPI 1.5.0',
                    #'Text Mined Cooccurrence API',
                    #'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    #'CATRAX Pharmacogenomics KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
#for api in APInames:
#    if 'Automat' in api and api not in selected_APIlist:
#        selected_APIlist.append(api)
        
#selected_APIlist = ['Retriever'] # select just Retriever endpoint
# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}


selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)


All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

## Find the neighborhood of an entity from a subset of APIs 


In [21]:
#name_resolver.lookup('BACE1', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')
#name_resolver.lookup('NPM1', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('4q12 microdeletion syndrome')
name_resolver.lookup('CDK9', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')
name_resolver.lookup('alzheimer disease', return_top_response=False, biolink_type='biolink:Disease',  limit=100) # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('Penicillamine')
#name_resolver.lookup('Penicillamine', return_top_response=True, biolink_type='biolink:Drug',  limit=100) # sometimes the identifiers are not in the top 1, users need to check the other returned results

#name_resolver.lookup('CDK9', return_top_response = False)

name_resolver.lookup('MYB', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')

name_resolver.lookup('ovarian cancer', return_top_response=True, biolink_type='biolink:Disease',  limit=10) 
name_resolver.lookup('acute myeloid leukemia', return_top_response=True, biolink_type='biolink:Disease',  limit=10) # sometimes the identifiers are not in the top 1, users need to check the other returned results



TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])

In [30]:
name_resolver.lookup('Myelodysplasia', return_top_response=False, biolink_type='biolink:Disease',  limit=50)

[TranslatorNode(curie='MONDO:0018881', label='Myelodysplasia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0013851', label='autosomal dominant aplasia and myelodysplasia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0009646', label='monosomy 7 myelodysplasia and leukemia syndrome 1', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0011043', label='myelodysplasia, immunodeficiency,

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [31]:

#input_identifiers = 'MONDO:0004975'
#
input_identifiers = 'MONDO:0016833'
#input_identifiers = 'CHEBI:145499'
input_identifiers = 'MONDO:0004975'
#input_identifiers = 'NCBIGene:1956'
input_node_info = node_normalizer.get_normalized_nodes(input_identifiers)
input_node_info
input_identifiers = "MONDO:0016833"
input_identifiers = "NCBIGene:4869"
#input_identifiers = 'MONDO:0018874'
input_identifiers = "NCBIGene:4869"
input_identifiers = 'MONDO:0016833' # 14q12 microdeletion syndrome 
input_identifiers = 'NCBIGene:2290' # FOXG1
input_identifiers = "NCBIGene:6261"
input_identifiers = "MONDO:0004975" # Alzheimer's disease
input_identifiers = "MONDO:0008170" # 14q12 microde
input_identifiers = "MONDO:0018874"
input_identifiers = "MONDO:0018881" #  Myelodysplasia


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [28]:
selected_metaKG

,API,Predicate,Subject,Object,URL
12848,Retriever,biolink:has_phenotype,biolink:Protein,biolink:PhenotypicFeature,https://retriever.ci.transltr.io/query/
12849,Retriever,biolink:has_phenotype,biolink:Protein,biolink:Disease,https://retriever.ci.transltr.io/query/
12850,Retriever,biolink:has_phenotype,biolink:Gene,biolink:Disease,https://retriever.ci.transltr.io/query/
12851,Retriever,biolink:has_substrate,biolink:Protein,biolink:Drug,https://retriever.ci.transltr.io/query/
12852,Retriever,biolink:has_substrate,biolink:Gene,biolink:Drug,https://retriever.ci.transltr.io/query/
...,...,...,...,...,...
16856,Retriever,biolink:affects,biolink:SmallMolecule,biolink:Gene,https://retriever.ci.transltr.io/query/
16857,Retriever,biolink:coexists_with,biolink:Gene,biolink:SmallMolecule,https://retriever.ci.transltr.io/query/
16858,Retriever,biolink:affects,biolink:Gene,biolink:SmallMolecule,https://retriever.ci.transltr.io/query/
16859,Retriever,biolink:interacts_with,biolink:Gene,biolink:SmallMolecule,https://retriever.ci.transltr.io/query/


In [32]:
# to exclude BioThings Explorer (BTE) TRAPI: 
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT_neighborhood_finder.neighborhood_finder(input_identifiers,
                                                                                            #node2_categories = ['biolink:Drug','biolink:SmallMolecule','biolink:ChemicalSubstance'],
                                                                                            #node2_categories = ['biolink:AnatomicalEntity'],
                                                                                            node2_categories = ['biolink:Gene'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

TCT_neighborhood_finder_result = TCT_neighborhood_finder.parse_results_for_neighborhood_finder(input_identifiers, result,
        start_node_categories='biolink:Gene',
          end_node_categories=None,
        get_node_info=True,
        scoring_method='infores')

# write a result to a json file
import json
# add a timestamp to the file name
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('TCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'_'+timestamp+'.json', 'w') as f:
    json.dump(TCT_neighborhood_finder_result, f)

MONDO:0018881
Retriever: Success!
